<a href="https://colab.research.google.com/github/AlexisRR13/BigData-33/blob/main/MNA_NLP_actividad_Busqueda_de_Talento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos: Sistema inteligente para búsqueda de talento.**

* **Nombre: Alexis Rodriguez Rodriguez**
* **Matrícula: A01797963**





* ##### **El formato de este cuaderno de Jupyter es libre, pero incluye al menos lo solicitado en el archivo PDF asociado a esta actividad.**

* ##### **Pueden importar los paquetes o librerías que requieran.**

* ##### **Pueden incluir las celdas y líneas de código que deseen.**


## 1. Introducción

En este proyecto se desarrolla un sistema inteligente basado en Procesamiento de Lenguaje Natural (NLP) que permite analizar, comparar y recomendar candidatos a partir de una vacante específica.

El sistema utiliza:
- Generación de CVs sintéticos
- Procesamiento multimodal (PDF + imágenes)
- Vectorización de perfiles
- Búsqueda semántica
- Ranking automático con justificación


In [57]:
#INSTALACION DE LIBRERIAS
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q transformers accelerate bitsandbytes
!pip install -q pypdf
!pip install -q pytesseract
!pip install -q pillow

In [58]:
#IMPORTACION DE LIBRERIAS
import os
import torch
import chromadb
import pandas as pd
import pytesseract

from PIL import Image
from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)


In [59]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import pipeline
#CARGA DE MODELO LLM PARA GENREAR CVS SINTETICOS
modelo_cv = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer_cv = AutoTokenizer.from_pretrained(modelo_cv)

model_cv = AutoModelForCausalLM.from_pretrained(
    modelo_cv,
    device_map="auto"
)

generador_cv = pipeline(
    "text-generation",
    model=model_cv,
    tokenizer=tokenizer_cv
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [60]:
#FUNCION PARA CREAR EL PROMPT DE CADA CV
def crear_prompt_cv(nivel, perfil):

    return f"""
Genera un CV sintético profesional.

Nivel: {nivel}

Perfil: {perfil}

NO incluyas:
- Nombre
- Fotografía
- Edad
- Género
- Nacionalidad
- Estado civil

Incluye:

1. Resumen profesional
2. Formación académica
3. Experiencia laboral
4. Habilidades técnicas
5. Habilidades blandas
6. Certificaciones
7. Idiomas
8. Proyectos relevantes

Debe ser realista y coherente con el perfil.

Entrega únicamente el CV.
"""

In [61]:
#DISTRIBUCION DE LOS 20 PERFILES SINTETICOS
perfiles = [

    ("Junior","Data Scientist"),
    ("Junior","Data Scientist"),
    ("Junior","BI Analyst"),
    ("Junior","Software Developer"),
    ("Junior","Hybrid AI Profile"),

    ("Intermedio","Data Engineer"),
    ("Intermedio","Data Engineer"),
    ("Intermedio","Data Engineer"),
    ("Intermedio","BI Analyst"),
    ("Intermedio","BI Analyst"),
    ("Intermedio","ML Engineer"),
    ("Intermedio","ML Engineer"),
    ("Intermedio","Software Architect"),
    ("Intermedio","Software Architect"),
    ("Intermedio","Hybrid AI Profile"),

    ("Senior","Data Scientist"),
    ("Senior","Data Scientist"),
    ("Senior","ML Engineer"),
    ("Senior","Software Architect"),
    ("Senior","Hybrid AI Profile")
]

In [62]:
#GENERACION DE LOS 20 CVS SINTETICOS
#CADA CV SE ALMACENA EN UNA LISTA DE DICCIONARIOS JUNTO CON SU NIVEL, PERFIL E ID
cvs = []

for nivel, perfil in perfiles:

    prompt = crear_prompt_cv(
        nivel,
        perfil
    )

    respuesta = generador_cv(
        prompt,
        max_new_tokens=1000,
        temperature=0.8,
        do_sample=True
    )

    texto_cv = respuesta[0]["generated_text"]

    cvs.append({
        "nivel": nivel,
        "perfil": perfil,
        "texto": texto_cv
    })

[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

In [63]:
#GUARDAR CVS EN ARCHIVOS TXT
import os

os.makedirs("CVs_Generados", exist_ok=True)

for i, cv in enumerate(cvs):

    archivo = f"CVs_Generados/cv_{i+1:02d}.txt"

    with open(
        archivo,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(cv["texto"])

In [64]:
!pip install reportlab

In [65]:
from reportlab.platypus import SimpleDocTemplate
from reportlab.platypus import Paragraph
from reportlab.lib.styles import getSampleStyleSheet

In [66]:
#CREACION DE PDFS TEMPORALES PARA LOS 20 CVS
os.makedirs("CVs_PDF", exist_ok=True)

styles = getSampleStyleSheet()

for i, cv in enumerate(cvs):

    pdf_file = f"CVs_PDF/cv_{i+1:02d}.pdf"

    doc = SimpleDocTemplate(pdf_file)

    contenido = [
        Paragraph(
            cv["texto"].replace("\n","<br/>"),
            styles["BodyText"]
        )
    ]

    doc.build(contenido)

In [67]:
junior_png = [0]
intermedio_png = [5,6]
senior_png = []

In [68]:
!pip install pdf2image
from pdf2image import convert_from_path

In [69]:
!apt-get update
!apt-get install poppler-utils

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,632 B in 2s (2,153 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [70]:
#CREACION DE CARPETAS FINALES DE BASE DE CONOCIMIENTO
#CONVERSION DE 4 PDFS A PNG
os.makedirs("CVs_PNG", exist_ok=True)

indices_png = [0,5,6,15]

for idx in indices_png:

    pdf_file = f"CVs_PDF/cv_{idx+1:02d}.pdf"

    paginas = convert_from_path(pdf_file)

    paginas[0].save(
        f"CVs_PNG/cv_{idx+1:02d}.png",
        "PNG"
    )

In [71]:
# LIBRERIAS

import os
import numpy as np
import pandas as pd

from pypdf import PdfReader

from PIL import Image
import pytesseract

from sentence_transformers import SentenceTransformer

import chromadb

In [72]:
# FUNCION PARA LEER PDFs

def leer_pdf(ruta_pdf):

    """
    Lee un archivo PDF completo y
    regresa todo el texto concatenado.
    """

    reader = PdfReader(ruta_pdf)

    texto = ""

    for pagina in reader.pages:

        contenido = pagina.extract_text()

        if contenido:
            texto += contenido + "\n"

    return texto

In [73]:
# OCR PARA PNG

def leer_png(ruta_png):

    """
    Extrae texto de una imagen PNG
    utilizando Tesseract OCR.
    """

    imagen = Image.open(ruta_png)

    texto = pytesseract.image_to_string(
        imagen,
        lang="eng"
    )

    return texto

In [74]:
# CARGAR PDFS

documentos = []
metadatos = []

ruta_pdf = "/content/CVs_PDF"

for archivo in os.listdir(ruta_pdf):

    if archivo.endswith(".pdf"):

        texto = leer_pdf(
            os.path.join(ruta_pdf, archivo)
        )

        documentos.append(texto)

        metadatos.append({
            "archivo": archivo,
            "tipo": "PDF"
        })

print(f"PDF cargados: {len(documentos)}")

PDF cargados: 20


In [75]:
# CARGAR PNGS

ruta_png = "/content/CVs_PNG"

for archivo in os.listdir(ruta_png):

    if archivo.endswith(".png"):

        texto = leer_png(
            os.path.join(ruta_png, archivo)
        )

        documentos.append(texto)

        metadatos.append({
            "archivo": archivo,
            "tipo": "PNG"
        })

print(f"Documentos totales: {len(documentos)}")

Documentos totales: 24


In [76]:
# DATAFRAME

df_cvs = pd.DataFrame(metadatos)

df_cvs["texto"] = documentos

df_cvs.head()

,archivo,tipo,texto
0,cv_16.pdf,PDF,Genera un CV sintético profesional.\nNivel: Se...
1,cv_05.pdf,PDF,Genera un CV sintético profesional.\nNivel: Ju...
2,cv_11.pdf,PDF,Genera un CV sintético profesional.\nNivel: In...
3,cv_02.pdf,PDF,Genera un CV sintético profesional.\nNivel: Ju...
4,cv_15.pdf,PDF,Genera un CV sintético profesional.\nNivel: In...


In [77]:
# EMBEDDINGS

embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-small"
)

print("Modelo cargado")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado


In [78]:
# CREACION DE EMBEDDINGS

embeddings = embedding_model.encode(
    documentos,
    show_progress_bar=True
)

print("Embeddings creados")
print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings creados
(24, 384)


In [79]:
# CHROMADB

client = chromadb.PersistentClient(
    path="./TalentVectorDB"
)

collection = client.get_or_create_collection(
    name="TalentDatabase"
)

print("Colección creada")

Colección creada


In [80]:
# INSERTAR CURRICULUMS

for i in range(len(documentos)):

    collection.add(

        ids=[f"CV_{i+1}"],

        documents=[documentos[i]],

        embeddings=[
            embeddings[i].tolist()
        ]
    )

print("CVs almacenados correctamente")


CVs almacenados correctamente


In [81]:
import numpy as np
# RECUPERACIÓN SEMÁNTICA

def recuperar_candidatos(
    descripcion_vacante,
    top_k=10
):

    # embedding de la vacante

    query_embedding = embedding_model.encode(
        [descripcion_vacante]
    )[0]

    resultados = collection.query(

        query_embeddings=[
            query_embedding.tolist()
        ],

        n_results=top_k,
        include = ['documents', 'metadatas']
    )

    return resultados

In [82]:
#VACANTE

vacante = """
Empresa: Microsoft

Puesto:
Data Scientist

Requisitos:

- Python
- SQL
- Machine Learning
- NLP
- Deep Learning
- Azure

Experiencia mínima:
3 años

Certificación:
Azure AI Engineer
"""


In [83]:
# RECUPERAR TOP 10

resultados = recuperar_candidatos(
    vacante,
    top_k=10
)

ids_recuperados = resultados["ids"][0]
docs_recuperados = resultados["documents"][0]

In [84]:
# LLM

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

modelo_llm = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    modelo_llm
)

In [85]:
# CONFIGURACION DE CUANTIZACION 4-BIT (QLoRA / BitsAndBytes)
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [86]:
# CARGA DEL MODELO DE LENGUAJE (LLM)
model = AutoModelForCausalLM.from_pretrained(

    modelo_llm,

    device_map="auto",

    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [87]:
# UNION DE LOS CVs RECUPERADOS

contexto = ""

for i, cv in enumerate(docs_recuperados):

    contexto += f"""

    ====================================================

    CANDIDATO {i+1}

    {cv}

    ====================================================

    """

In [88]:
# GENERACIÓN

inputs = tokenizer(

    prompt,

    return_tensors="pt",

    truncation=True,

    max_length=4096

).to(model.device)


In [89]:
# GENERACIÓN DE RESPUESTA CON EL MODELO LLM
salida = model.generate(

    **inputs,

    max_new_tokens=1200,

    do_sample=True,

    temperature=0.2
)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [90]:
# DECODIFICACION DE LA RESPUESTA GENERADA
respuesta = tokenizer.decode(

    salida[0],

    skip_special_tokens=True
)

print(respuesta)


Genera un CV sintético profesional.

Nivel: Senior

Perfil: Hybrid AI Profile

NO incluyas:
- Nombre
- Fotografía
- Edad
- Género
- Nacionalidad
- Estado civil

Incluye:

1. Resumen profesional
2. Formación académica
3. Experiencia laboral
4. Habilidades técnicas
5. Habilidades blandas
6. Certificaciones
7. Idiomas
8. Proyectos relevantes

Debe ser realista y coherente con el perfil.

Entrega únicamente el CV.
```json
{
  "Resumen": "Soy un experto en la integración de inteligencia artificial, especializado en el desarrollo de algoritmos para procesamiento del lenguaje natural y aprendizaje automático.",
  "FormacionAcademica": [
    {
      "NombreDeLaUniversidad": "Universidad XYZ",
      "Grado": "Master en Inteligencia Artificial",
      "AñoDeConclusión": "2023"
    },
    {
      "NombreDeLaUniversidad": "Instituto ABC",
      "Grado": "Licenciatura en Informática",
      "AñoDeConclusión": "2019"
    }
  ],
  "ExperienciaLaboral": [
    {
      "Empresa": "Proyecto XYZ",
      

In [91]:
# GUARDAR RESULTADOS
with open(
    "Ranking_Top5.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(respuesta)

print("Ranking guardado")

Ranking guardado


In [92]:
# ESTADÍSTICAS

print("CVs procesados")

CVs procesados


Arquitectura Implementada

La solucion desarrollada corresponde a un sistema inteligente de búsqueda de talento basado en Procesamiento de Lenguaje Natural, generación sintética de datos, procesamiento multimodal, embeddings, base de datos vectorial y generación aumentada por recuperación. La actividad solicita construir un sistema capaz de analizar CVs, compararlos contra una vacante, recuperar información relevante desde una base vectorial construida con archivos PDF y PNG, y jerarquizar a los 5 mejores candidatos con justificación.

# **Conclusiones:**

* **Coinsidero que esta actividad me permitió construir un sistema inteligente de búsqueda de talento que integra varias etapas relevantes de un pipeline moderno de NLP aplicado a reclutamiento. La solución incluye generación de CVs sintéticos, procesamiento de documentos en PDF e imagen, extracción de texto, representación vectorial, búsqueda semántica y generación de recomendaciones justificadas mediante un LLM.
Uno de los aspectos más importantes fue el uso de CVs sintéticos, ya que permite desarrollar y probar la solución sin exponer información personal real. Además, evitar datos como nombre, edad, género, fotografía, nacionalidad o estado civil ayuda a reducir sesgos y enfocar el análisis en competencias, experiencia, formación y proyectos.
Desde el punto de vista técnico, la arquitectura RAG con ChromaDB resulta adecuada para este problema, porque permite recuperar perfiles similares a la vacante antes de pedir al LLM una evaluación final. Esto mejora la eficiencia del sistema, ya que el modelo no analiza todos los documentos al mismo tiempo, sino solo aquellos más relevantes.
La solución también demuestra el valor de las bases vectoriales para escenarios de recursos humanos, especialmente cuando la búsqueda por palabras clave puede ser insuficiente. Por ejemplo, un candidato puede no escribir exactamente “NLP”, pero sí mencionar procesamiento de texto, modelos de lenguaje o clasificación semántica. Los embeddings ayudan a capturar este tipo de relaciones.
Como conclusión, el sistema desarrollado cumple el objetivo de apoyar la selección inicial de candidatos. No sustituye al reclutador ni al experto técnico, pero sí puede reducir el tiempo de revisión, mejorar la trazabilidad de los criterios utilizados y ofrecer una primera recomendación razonada. Para una implementación real, sería recomendable agregar validación humana, métricas formales de desempeño, control de sesgos, auditoría de resultados y pruebas con vacantes reales adicionales, Asi como datos especificos que busquen los reclutadores.**


# **Fin de la actividad : Búsqueda Inteligente de Talento**